# Module 04 — Deux conteneurs qui se parlent

**Formation Big Data — ANSD / Data Innovation Lab**

Ce notebook ne s'exécute pas sur votre machine : il tourne **dans un
conteneur**. Et la base de données à laquelle il va se connecter tourne dans un
**autre conteneur**. Aucun des deux n'est installé sur votre poste.

Au programme :

1. vérifier où l'on se trouve ;
2. se connecter à PostgreSQL — sans mot de passe écrit en clair ;
3. créer une base, deux tables, y insérer des données ;
4. relire pour vérifier ;
5. constater ce qui survit à l'arrêt des conteneurs.

## 1. Où sommes-nous ?

In [ ]:
import os
import platform

print("Machine       :", platform.node())
print("Système       :", platform.platform())
print("Dossier       :", os.getcwd())
print("Dans Docker ? :", os.path.exists("/.dockerenv"))

Le nom de machine est un identifiant engendré par Docker, et le fichier
`/.dockerenv` signale sans ambiguïté que nous sommes dans un conteneur.

Ce notebook est pourtant enregistré **sur votre disque**, dans le dossier
`notebooks` que vous avez monté : c'est tout l'intérêt du volume. Le conteneur
peut disparaître, votre travail reste.

## 2. Se connecter à la base

Les identifiants ne sont écrits nulle part dans ce notebook : ils ont été
transmis au conteneur par le fichier `.env`, via `docker compose`. On les lit
dans les variables d'environnement.

In [ ]:
from sqlalchemy import create_engine, text

UTILISATEUR = os.environ["POSTGRES_USER"]
MOT_DE_PASSE = os.environ["POSTGRES_PASSWORD"]
HOTE = os.environ["POSTGRES_HOTE"]      # ← regardez bien cette valeur
PORT = os.environ["POSTGRES_PORT"]

print("Utilisateur :", UTILISATEUR)
print("Mot de passe :", "*" * len(MOT_DE_PASSE))
print("Hôte :", HOTE, "| Port :", PORT)

**Question 1.** L'hôte de la base n'est ni `localhost`, ni une adresse IP,
mais `postgres`. D'où vient ce nom, et comment ce conteneur peut-il le
résoudre ?

*Votre réponse :* …

> C'est le **réseau Docker**. `compose` a créé un réseau privé entre les deux
> services et y a inscrit chacun sous le nom que vous lui avez donné dans le
> fichier `docker-compose.yml`. Les conteneurs s'appellent par leur nom de
> service. 

In [ ]:
# On se connecte d'abord à la base « postgres », créée par défaut.
# `AUTOCOMMIT` est nécessaire : PostgreSQL refuse de créer une base à
# l'intérieur d'une transaction.
moteur_admin = create_engine(
    f"postgresql+psycopg2://{UTILISATEUR}:{MOT_DE_PASSE}@{HOTE}:{PORT}/postgres",
    isolation_level="AUTOCOMMIT",
)

with moteur_admin.connect() as connexion:
    version = connexion.execute(text("SELECT version()")).scalar()
print(version)

## 3. Créer la base `school`

In [ ]:
# Créez la base de données `school`.
#
# Indices :
#   - relancer la cellule provoquera une erreur « database already exists » :
#     c'est normal, et c'est même une bonne occasion de lire un message d'erreur.

with moteur_admin.connect() as connexion:
    connexion.execute(
        text("CREATE DATABASE school")
    )

# Vérification : la base figure-t-elle dans la liste ?
with moteur_admin.connect() as connexion:
    bases = connexion.execute(
        text("SELECT datname FROM pg_database WHERE datistemplate = false")
    ).scalars().all()
print(bases)

In [ ]:
# On se reconnecte, cette fois sur la base `school`
moteur = create_engine(
    f"postgresql+psycopg2://{UTILISATEUR}:{MOT_DE_PASSE}@{HOTE}:{PORT}/school"
)

with moteur.connect() as connexion:
    print("Connecté à :", connexion.execute(text("SELECT current_database()")).scalar())

## 4. Créer les deux tables

Nous modélisons une petite scolarité, aux couleurs de l'ENSAE Dakar :

- `etudiants` — nom, date de naissance, classe
- `cours` — intitulé, durée en heures, niveau

In [ ]:
# Créez les deux tables.
#
# Colonnes attendues :
#   etudiants : id (SERIAL PRIMARY KEY), nom (TEXT), date_naissance (DATE),
#               classe (TEXT)
#   cours     : id (SERIAL PRIMARY KEY), intitule (TEXT), duree_heures (INTEGER),
#               niveau (TEXT)
#
creation_etudiants = """
CREATE TABLE IF NOT EXISTS etudiants (
    id SERIAL PRIMARY KEY,
    nom TEXT,
    date_naissance DATE,
    classe TEXT
);
"""

creation_cours = """
CREATE TABLE IF NOT EXISTS cours (
    id SERIAL PRIMARY KEY,
    intitule TEXT,
    duree_heures INTEGER,
    niveau TEXT
);
"""

with moteur.connect() as connexion:
    connexion.execute(text(creation_etudiants))
    connexion.execute(text(creation_cours))
    connexion.commit()

print("Tables créées.")

In [ ]:
# Vérification : quelles tables existent dans la base ?
import pandas as pd

pd.read_sql(
    """SELECT table_name, column_name, data_type
       FROM information_schema.columns
       WHERE table_schema = 'public'
       ORDER BY table_name, ordinal_position""",
    moteur,
)

## 5. Insérer des données

In [ ]:
# Fourni : les référentiels
import numpy as np

NOMS = ["Diop", "Ndiaye", "Fall", "Sarr", "Ba", "Sow", "Diallo", "Gueye",
        "Faye", "Sy", "Cissé", "Mbaye", "Seck", "Thiam", "Dieng", "Camara",
        "Sagna", "Badji", "Diatta", "Mané", "Diouf", "Kane", "Touré", "Samb"]
PRENOMS = ["Mamadou", "Aminata", "Ousmane", "Fatou", "Ibrahima", "Awa",
           "Cheikh", "Ndeye", "Moussa", "Astou", "Abdoulaye", "Khadija",
           "Alioune", "Seynabou", "Babacar", "Coumba", "Lamine", "Bineta"]
CLASSES = ["AS1", "AS2", "ITS1", "ITS2", "ITS3", "ISE1", "ISE2", "ISE3"]

COURS = [
    ("Probabilités", 60, "ITS1"),
    ("Statistique inférentielle", 45, "ITS2"),
    ("Économétrie", 50, "ISE1"),
    ("Séries temporelles", 40, "ISE2"),
    ("Théorie des sondages", 55, "ITS2"),
    ("Comptabilité nationale", 35, "ISE1"),
    ("Analyse de données", 45, "ITS3"),
    ("Bases de données", 30, "ITS1"),
    ("Programmation Python", 40, "AS2"),
    ("Apprentissage automatique", 50, "ISE3"),
    ("Démographie", 35, "ITS1"),
    ("Enquêtes ménages", 30, "AS1"),
]

generateur = np.random.default_rng(2026)
print(f"{len(NOMS)} noms, {len(PRENOMS)} prénoms, {len(CLASSES)} classes, "
      f"{len(COURS)} cours de référence.")

In [ ]:
# Engendrez 200 étudiants et insérez-les, puis insérez
# les cours.
#
# Pour chaque étudiant : un nom complet (« Prénom Nom »), une date de naissance
# tirée entre 1998 et 2006, une classe tirée au hasard.
#

etudiants = pd.DataFrame({
    "nom": (
        generateur.choice(PRENOMS, size=200)
        + " "
        + generateur.choice(NOMS, size=200)
    ),
    "date_naissance": pd.to_datetime({
        "year": generateur.integers(1998, 2007, size=200),
        "month": generateur.integers(1, 13, size=200),
        "day": generateur.integers(1, 29, size=200)  # 1 à 28 pour éviter les dates invalides
    }),
    "classe": generateur.choice(CLASSES, size=200),
})

cours = pd.DataFrame(COURS, columns=["intitule", "duree_heures", "niveau"])

etudiants.to_sql("etudiants", moteur, if_exists="append", index=False)
cours.to_sql("cours", moteur, if_exists="append", index=False)

print("Insertion terminée.")

## 6. Vérifier

In [ ]:
# Relisez les deux tables et vérifiez l'insertion :
#   - combien d'étudiants, combien de cours ?
#   - la répartition des étudiants par classe
#   - le volume horaire total par niveau, du plus chargé au moins chargé
#
# Lecture des tables
df_etudiants = pd.read_sql("SELECT * FROM etudiants", moteur)
df_cours = pd.read_sql("SELECT * FROM cours", moteur)

# Nombre d'étudiants et de cours
print("Nombre d'étudiants :", len(df_etudiants))
print("Nombre de cours :", len(df_cours))

# Répartition des étudiants par classe
print("\nRépartition des étudiants par classe :")
repartition_classes = (
    df_etudiants["classe"]
    .value_counts()
    .reset_index()
)

repartition_classes.columns = ["classe", "nombre_etudiants"]
print(repartition_classes)

# Volume horaire total par niveau, du plus chargé au moins chargé
print("\nVolume horaire total par niveau :")
volume_par_niveau = (
    df_cours
    .groupby("niveau")["duree_heures"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

volume_par_niveau.columns = ["niveau", "volume_total_heures"]
print(volume_par_niveau)

In [ ]:
# Fourni : une jointure entre les deux tables, pour finir
pd.read_sql(
    """SELECT e.classe,
              COUNT(DISTINCT e.id)  AS nb_etudiants,
              COUNT(DISTINCT c.id)  AS nb_cours,
              COALESCE(SUM(c.duree_heures), 0) AS heures
       FROM etudiants e
       LEFT JOIN cours c ON c.niveau = e.classe
       GROUP BY e.classe
       ORDER BY e.classe""",
    moteur,
)

## 7. Ce qui survit, et ce qui disparaît

*Démonstration à faire dans le terminal, pas ici.*

Vos données sont-elles dans le conteneur, ou dans le volume ? Vérifions.

**Première expérience — arrêter et relancer**

```bash
docker compose down          # les conteneurs sont supprimés
docker compose up -d         # on relance
```

Revenez ensuite dans ce notebook, réexécutez la cellule de connexion et la
lecture des tables. Les données sont toujours là : elles vivent dans le
**volume**, pas dans le conteneur.

**Seconde expérience — supprimer aussi les volumes**

```bash
docker compose down -v       # attention au -v
docker compose up -d
```

Cette fois, la base `school` a disparu. Il faudra tout recréer.

**Question 2.** Formulez en une phrase la différence entre un conteneur et un
volume. Et retenez ce que fait l'option `-v` — c'est une commande à manipuler
avec prudence.

*Votre réponse :* …

## 8. Ce qu'il faut retenir

- Un conteneur est **jetable** ; un volume **persiste**. Toute donnée qui
  compte doit vivre dans un volume.
- Deux conteneurs d'une même pile se joignent par leur **nom de service**, sur
  un réseau privé créé par `compose`.
- Les identifiants passent par un fichier `.env`, jamais par le code, jamais
  dans Git.
- Un fichier `docker-compose.yml` remplace une longue série de commandes, et il
  se versionne : votre environnement devient reproductible.
- Rien de tout cela n'a été installé sur votre poste. À l'exception du moteur de
  conteneurs, votre machine est restée intacte.

**À compléter :**

- Nombre d'étudiants insérés : …
- Classe la plus nombreuse : …
- Ce qui a disparu après `docker compose down -v` : …
